# Getting Started with the OpenAI Agents SDK

The [OpenAI Agents SDK](https://github.com/openai/openai-agents-python) is a lightweight framework for building multi-agent workflows. This notebook walks you through everything from a basic agent to a multi-agent pipeline with streaming, handoffs, and guardrails.

**What you'll learn:**
1. Creating an agent and running it
2. Giving agents tools (function calling)
3. Structured outputs with Pydantic
4. Streaming responses in real time
5. Handoffs — agents routing to other agents
6. Input guardrails — validating user input before the agent runs
7. Putting it all together in a real example

## Setup

In [ ]:
%pip install openai-agents --quiet

In [ ]:
import asyncio
import os

from agents import (
    Agent,
    GuardrailFunctionOutput,
    InputGuardrail,
    Runner,
    function_tool,
    handoff,
)
from pydantic import BaseModel

os.environ.setdefault("OPENAI_API_KEY", "your-api-key-here")

## 1. Your First Agent

An `Agent` is just an AI model with a name, instructions, and optional tools. `Runner.run()` executes it and returns a result.

In [ ]:
agent = Agent(
    name="Assistant",
    instructions="You are a helpful assistant. Answer concisely.",
    model="gpt-4o-mini",
)

result = await Runner.run(agent, "What is the capital of France?")
print(result.final_output)

## 2. Giving the Agent Tools

Use `@function_tool` to expose Python functions to the agent. The agent decides when to call them based on the user's request.

In [ ]:
@function_tool
def get_weather(city: str) -> str:
    """Get the current weather for a city.

    Args:
        city: Name of the city.
    """
    # In a real app, you'd call a weather API here
    weather_data = {
        "London": "Cloudy, 15°C",
        "Tokyo": "Sunny, 28°C",
        "New York": "Rainy, 18°C",
    }
    return weather_data.get(city, f"No data available for {city}")


@function_tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount from one currency to another.

    Args:
        amount: The amount to convert.
        from_currency: Source currency code (e.g. USD).
        to_currency: Target currency code (e.g. EUR).
    """
    # Simplified mock rates
    rates = {"USD": 1.0, "EUR": 0.92, "GBP": 0.79, "JPY": 149.5}
    if from_currency not in rates or to_currency not in rates:
        return f"Unknown currency. Supported: {', '.join(rates)}"
    converted = amount / rates[from_currency] * rates[to_currency]
    return f"{amount} {from_currency} = {converted:.2f} {to_currency}"


travel_agent = Agent(
    name="Travel Assistant",
    instructions="Help users with travel questions. Use tools to get weather and currency info.",
    tools=[get_weather, convert_currency],
    model="gpt-4o-mini",
)

result = await Runner.run(
    travel_agent,
    "I'm traveling to Tokyo. What's the weather like, and how much is 100 USD in JPY?",
)
print(result.final_output)

## 3. Structured Outputs

Pass a Pydantic model as `output_type` and the agent will always return a validated, typed object.

In [ ]:
class TravelPlan(BaseModel):
    destination: str
    duration_days: int
    top_activities: list[str]
    estimated_budget_usd: float
    best_time_to_visit: str


planner = Agent(
    name="Travel Planner",
    instructions="Create detailed travel plans based on user preferences.",
    output_type=TravelPlan,
    model="gpt-4o-mini",
)

result = await Runner.run(planner, "Plan a 5-day trip to Kyoto, Japan on a moderate budget.")
plan: TravelPlan = result.final_output

print(f"Destination     : {plan.destination}")
print(f"Duration        : {plan.duration_days} days")
print(f"Budget          : ${plan.estimated_budget_usd:,.0f}")
print(f"Best time       : {plan.best_time_to_visit}")
print(f"Top activities  :")
for activity in plan.top_activities:
    print(f"  - {activity}")

## 4. Streaming

Use `Runner.run_streamed()` to get text as it is generated rather than waiting for the full response.

In [ ]:
from agents import RunResultStreaming
from agents.stream_events import RawResponsesStreamEvent
from openai.types.responses import ResponseTextDeltaEvent

writer_agent = Agent(
    name="Writer",
    instructions="Write engaging content on any topic.",
    model="gpt-4o-mini",
)

print("Streaming response:\n")
async with Runner.run_streamed(writer_agent, "Write a short poem about the ocean.") as stream:
    async for event in stream.stream_events():
        if isinstance(event, RawResponsesStreamEvent):
            if isinstance(event.data, ResponseTextDeltaEvent):
                print(event.data.delta, end="", flush=True)

print("\n\n[Stream complete]")

## 5. Handoffs — Routing Between Agents

An agent can hand off to another specialized agent mid-conversation. The receiving agent takes over with the full conversation context.

In [ ]:
# Specialist agents
flight_agent = Agent(
    name="Flight Specialist",
    instructions="You are an expert on flights and aviation. Answer flight-related questions thoroughly.",
    model="gpt-4o-mini",
)

hotel_agent = Agent(
    name="Hotel Specialist",
    instructions="You are an expert on hotels and accommodations. Give detailed hotel recommendations.",
    model="gpt-4o-mini",
)

# Frontline agent that routes to the right specialist
concierge = Agent(
    name="Travel Concierge",
    instructions=(
        "You are a travel concierge. Route the user to the right specialist:\n"
        "- Flight questions → Flight Specialist\n"
        "- Hotel questions → Hotel Specialist\n"
        "- General questions → answer yourself"
    ),
    handoffs=[flight_agent, hotel_agent],
    model="gpt-4o-mini",
)

result = await Runner.run(concierge, "What's the best way to find cheap business class flights to Europe?")
print(f"Handled by: {result.last_agent.name}")
print(f"Response: {result.final_output}")

## 6. Input Guardrails

Guardrails run in parallel with the agent to validate or filter user input. If the guardrail trips, the agent stops immediately without wasting tokens.

In [ ]:
from agents import GuardrailTripwireTriggered, InputGuardrailTripwireTriggered


class TopicCheckOutput(BaseModel):
    is_travel_related: bool
    reason: str


topic_checker = Agent(
    name="Topic Checker",
    instructions="Determine if the user's message is travel-related. Output JSON.",
    output_type=TopicCheckOutput,
    model="gpt-4o-mini",
)


@InputGuardrail
async def travel_only_guardrail(ctx, agent, input_text) -> GuardrailFunctionOutput:
    result = await Runner.run(topic_checker, input_text)
    check: TopicCheckOutput = result.final_output
    return GuardrailFunctionOutput(
        output_info=check,
        tripwire_triggered=not check.is_travel_related,
    )


guarded_agent = Agent(
    name="Travel-Only Assistant",
    instructions="Help users with travel questions only.",
    input_guardrails=[travel_only_guardrail],
    model="gpt-4o-mini",
)

# Travel question — should pass
try:
    result = await Runner.run(guarded_agent, "What documents do I need to travel to Japan?")
    print(f"Travel question passed: {result.final_output[:100]}...")
except InputGuardrailTripwireTriggered:
    print("Travel question was incorrectly blocked.")

# Off-topic question — should be blocked
try:
    result = await Runner.run(guarded_agent, "What's the best Python sorting algorithm?")
    print(f"Off-topic question passed (unexpected): {result.final_output}")
except InputGuardrailTripwireTriggered as exc:
    print(f"Off-topic question correctly blocked by guardrail.")

## 7. Complete Example — Travel Assistant Pipeline

Combining everything: a guarded multi-agent travel assistant with structured output.

In [ ]:
class TripSummary(BaseModel):
    destination: str
    highlights: list[str]
    practical_tips: list[str]


summary_agent = Agent(
    name="Trip Summarizer",
    instructions="Summarize travel information into a clean, structured trip summary.",
    output_type=TripSummary,
    model="gpt-4o-mini",
)

info_agent = Agent(
    name="Travel Info Agent",
    instructions="Provide comprehensive travel information including highlights and practical tips.",
    handoffs=[summary_agent],
    tools=[get_weather, convert_currency],
    model="gpt-4o-mini",
)

result = await Runner.run(info_agent, "Tell me everything I need to know about visiting Lisbon, Portugal.")

if isinstance(result.final_output, TripSummary):
    summary = result.final_output
    print(f"Destination: {summary.destination}")
    print("\nHighlights:")
    for h in summary.highlights:
        print(f"  • {h}")
    print("\nPractical Tips:")
    for t in summary.practical_tips:
        print(f"  • {t}")
else:
    print(result.final_output)

## Summary

| Concept | SDK Feature | When to use |
|---|---|---|
| Basic agent | `Agent` + `Runner.run()` | Any task you'd describe to a person |
| Tool calling | `@function_tool` | When the agent needs real-time data or actions |
| Structured output | `output_type=PydanticModel` | When you need typed, validated responses |
| Streaming | `Runner.run_streamed()` | UIs, chat interfaces, long responses |
| Routing | `handoffs=[agent1, agent2]` | Specialized agents for different domains |
| Validation | `@InputGuardrail` | Filtering off-topic or harmful inputs |

### Next steps
- See `parallel_agents.ipynb` for running multiple agents concurrently
- See `session_memory.ipynb` for persistent multi-turn conversations
- See `tracking_token_usage_and_costs.ipynb` for cost monitoring
- See the [Agents SDK docs](https://openai.github.io/openai-agents-python/) for the full reference